# Scenario: Cross-Referencing AI Orders with Patient Allergies

In [13]:
import pandas as pd
import sqlite3
# 1. creating Table of AI-Suggested Medication Orders
orders_data = {
    "order_id": [101, 102, 103, 104],
    "patient_id": ["P-01", "P-02", "P-03", "P-04"],
    "suggested_med": ["Amoxicillin", "Ibuprofen", "Penicillin", "Aspirin"]    
}
# 2. creating Table of Known Patient Allergies
allergy_data = {
    "patient_id": ["P-01", "P-03", "P-05"],
    "allergy": ["Peanuts", "Penicillin", "Sulfa"]    
}
# adding data into dataFrame
df_orders = pd.DataFrame(orders_data)
df_allergy = pd.DataFrame(allergy_data)
# conneting to sql to save the data in teporary memory
connt = sqlite3.connect(":memory:")
# saving dataset into sql
df_orders.to_sql("orders", connt, index = False, if_exists = "replace")
df_allergy.to_sql("allergies", connt, index = False, if_exists = "replace")
# function to execute the query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("Relational Database (Orders & Allergies) is ready!")

Relational Database (Orders & Allergies) is ready!


# The "Never Event" Audit (Using an INNER JOIN)

In [14]:
# query for all dataset for a quick view
all_data_orders = "SELECT * FROM orders"
print("************************************* orders data table ***********************")
display(run_query(all_data_orders))
print()
all_data_allergies = "SELECT * FROM allergies"
print("********************************** allergies data table *************************")
display(run_query(all_data_allergies))
print()
# query that joins the orders table and the allergies table on the patient_id column(suggested_med should be the same as allergy)
cross_check = """
SELECT orders.patient_id, orders.suggested_med FROM orders
INNER JOIN allergies ON orders.patient_id = allergies.patient_id
WHERE orders.suggested_med = allergies.allergy
"""
print("******************************* cross_check result ***************************")
display(run_query(cross_check))


************************************* orders data table ***********************


,order_id,patient_id,suggested_med
0,101,P-01,Amoxicillin
1,102,P-02,Ibuprofen
2,103,P-03,Penicillin
3,104,P-04,Aspirin



********************************** allergies data table *************************


,patient_id,allergy
0,P-01,Peanuts
1,P-03,Penicillin
2,P-05,Sulfa



******************************* cross_check result ***************************


,patient_id,suggested_med
0,P-03,Penicillin


# List All Patients and Their Allergies (Using a LEFT JOIN)

In [17]:
# query to show all orders, and if the patient has a known allergy, show it in the next column
patient_allergies = """
SELECT orders.patient_id, orders.suggested_med, allergies.allergy FROM orders
LEFT JOIN allergies ON allergies.patient_id = orders.patient_id
"""
print("******************************* patients with allergies *****************")
display(run_query(patient_allergies))

******************************* patients with allergies *****************


,patient_id,suggested_med,allergy
0,P-01,Amoxicillin,Peanuts
1,P-02,Ibuprofen,None
2,P-03,Penicillin,Penicillin
3,P-04,Aspirin,None
